In [1]:
import sys
!{sys.executable} -m pip install --user transformers peft evaluate accelerate jiwer soundfile librosa
!{sys.executable} -m pip install --user "datasets==2.21.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys
sys.path.insert(0, '/home/jovyan/.local/lib/python3.11/site-packages')

import torch
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import load_dataset, load_from_disk
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

processor = AutoProcessor.from_pretrained("KBLab/kb-whisper-large")
processor.tokenizer.set_prefix_tokens(language="no", task="transcribe")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print('bf16:', torch.cuda.is_bf16_supported())

model_kb = AutoModelForSpeechSeq2Seq.from_pretrained(
    "KBLab/kb-whisper-large",
    dtype=torch.float16
).to(device)

# Load preprocessed data
ds_train = load_from_disk("/home/jovyan/nst_train_preprocessed")
ds_test  = load_from_disk("/home/jovyan/nst_test_preprocessed")


SyntaxError: unmatched ')' (3073690889.py, line 26)

In [ ]:
# Configure LoRA
# Whisper's attention projections are the standard LoRA target for ASR
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32,                        # rank — higher = more capacity, more params
    lora_alpha=64,               # scaling factor, commonly set to 2*r
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],   # encoder + decoder attention
    bias="none",
)

model_kb = get_peft_model(model_kb, lora_config)
model_kb.print_trainable_parameters()

In [ ]:
# ── 3. DATA COLLATOR ───────────────────────────────────────────────────────────
# Whisper needs a custom collator: audio features are padded to 30s (fixed),
# but label sequences need padding with -100 so loss ignores pad tokens.
@dataclass
class WhisperDataCollator:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = WhisperDataCollator(processor=processor)

In [ ]:
# ── 4. EVALUATION METRIC ───────────────────────────────────────────────────────
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
# ── 5. TRAINING ARGUMENTS ──────────────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir="./kb-whisper-lora-no",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,     # effective batch size = 16
    learning_rate=1e-4,
    warmup_steps=500,
    max_steps=5000,                    # adjust to dataset size / compute budget
    gradient_checkpointing=True,       # saves VRAM at cost of slightly slower step
    bf16=True,                         # use bf16=True if on Ampere GPU (A100 etc.)
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=448,
    report_to="none",                  # swap to "wandb" if you want experiment tracking
)

# ── 6. TRAINER ─────────────────────────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model=model_kb,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor_kb.feature_extractor,
)

In [ ]:
# ── 7. TRAIN ───────────────────────────────────────────────────────────────────
trainer.train()

# ── 8. SAVE ────────────────────────────────────────────────────────────────────
# Saves only the LoRA adapter weights — much smaller than the full model
model_kb.save_pretrained("./kb-whisper-lora-no")
processor.save_pretrained("./kb-whisper-lora-no")
print("Saved.")
